# Decision Tree Pipeline with Hyperparameter Tuning

This notebook demonstrates a complete machine learning pipeline using scikit-learn that:
1. Reads training and test data from CSV files
2. Preprocesses the data (handling missing values, encoding, scaling)
3. Builds a pipeline that feeds preprocessed data to a decision tree
4. Compares different decision tree variants to find the ideal type
5. Performs hyperparameter tuning using cross-validation

## 1. Imports and Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Decision Tree Models
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Data

Load training and test data from CSV files. Update the file paths as needed for your dataset.

In [ ]:
# Define file paths - UPDATE THESE FOR YOUR DATA
TRAIN_FILE = 'data/train.csv'
TEST_FILE = 'data/test.csv'

# Load the data
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

In [ ]:
# Preview the training data
train_df.head()

In [ ]:
# Basic info about the dataset
train_df.info()

In [ ]:
# Statistical summary
train_df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
missing_values = train_df.isnull().sum()
missing_pct = (missing_values / len(train_df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing %': missing_pct
}).sort_values(by='Missing %', ascending=False)

print("Missing Values Summary:")
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Identify column types
numerical_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Visualize distributions of numerical features
fig, axes = plt.subplots(nrows=(len(numerical_cols) + 2) // 3, ncols=3, figsize=(15, 4 * ((len(numerical_cols) + 2) // 3)))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    axes[idx].hist(train_df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

# Hide empty subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical features
if len(numerical_cols) > 1:
    plt.figure(figsize=(12, 10))
    correlation_matrix = train_df[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Feature Correlation Heatmap')
    plt.tight_layout()
    plt.show()

## 4. Data Preprocessing

### 4.1 Define Target and Features

In [ ]:
# Define target column - UPDATE THIS FOR YOUR DATA
TARGET_COL = 'target'  # Change this to your target column name

# Determine if this is a classification or regression problem
if train_df[TARGET_COL].dtype == 'object' or train_df[TARGET_COL].nunique() < 20:
    TASK_TYPE = 'classification'
    print(f"Task Type: Classification (Target has {train_df[TARGET_COL].nunique()} unique values)")
else:
    TASK_TYPE = 'regression'
    print(f"Task Type: Regression (Target is continuous)")

# Separate features and target
X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL]

# If classification with string labels, encode them
if TASK_TYPE == 'classification' and y.dtype == 'object':
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)
    print(f"Classes: {label_encoder.classes_}")

In [ ]:
# Update feature column lists (excluding target)
feature_cols = [col for col in X.columns]
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Total features: {len(feature_cols)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

### 4.2 Build Preprocessing Pipeline

In [ ]:
# Numerical preprocessing: impute missing values + scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing: impute missing values + one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessors using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'  # Keep any other columns as-is
)

print("Preprocessing pipeline created successfully!")

### 4.3 Train-Validation Split

In [ ]:
# Split training data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y if TASK_TYPE == 'classification' else None
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

## 5. Decision Tree Model Comparison

We'll compare different types of decision tree models to find the best one:
1. **Single Decision Tree** - Basic tree model
2. **Random Forest** - Bagging ensemble of trees
3. **Gradient Boosting** - Sequential boosting ensemble
4. **AdaBoost** - Adaptive boosting ensemble

In [ ]:
# Define models based on task type
if TASK_TYPE == 'classification':
    models = {
        'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
        'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=100),
        'AdaBoost': AdaBoostClassifier(random_state=RANDOM_STATE, n_estimators=100, algorithm='SAMME')
    }
    scoring = 'accuracy'
else:
    models = {
        'Decision Tree': DecisionTreeRegressor(random_state=RANDOM_STATE),
        'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100),
        'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE, n_estimators=100),
        'AdaBoost': AdaBoostRegressor(random_state=RANDOM_STATE, n_estimators=100)
    }
    scoring = 'neg_mean_squared_error'

print(f"Models to compare: {list(models.keys())}")
print(f"Scoring metric: {scoring}")

In [ ]:
# Compare models using cross-validation
model_results = {}

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    
    # Create pipeline with preprocessor and model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Perform cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring=scoring)
    
    # Store results
    model_results[name] = {
        'mean_score': cv_scores.mean(),
        'std_score': cv_scores.std(),
        'all_scores': cv_scores
    }
    
    if TASK_TYPE == 'classification':
        print(f"  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    else:
        rmse = np.sqrt(-cv_scores.mean())
        print(f"  CV RMSE: {rmse:.4f} (+/- {np.sqrt(-cv_scores).std() * 2:.4f})")

In [ ]:
# Visualize model comparison
plt.figure(figsize=(10, 6))

names = list(model_results.keys())
means = [model_results[name]['mean_score'] for name in names]
stds = [model_results[name]['std_score'] for name in names]

if TASK_TYPE == 'regression':
    # Convert negative MSE to RMSE for visualization
    means = [np.sqrt(-m) for m in means]
    ylabel = 'RMSE (lower is better)'
else:
    ylabel = 'Accuracy (higher is better)'

bars = plt.bar(names, means, yerr=stds, capsize=5, alpha=0.7, color='steelblue')
plt.xlabel('Model')
plt.ylabel(ylabel)
plt.title('Decision Tree Model Comparison (5-Fold CV)')
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar, mean in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{mean:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Determine the best model type
if TASK_TYPE == 'classification':
    best_model_name = max(model_results, key=lambda x: model_results[x]['mean_score'])
else:
    best_model_name = max(model_results, key=lambda x: model_results[x]['mean_score'])  # Less negative = better

print(f"\nBest Model Type: {best_model_name}")
print(f"Score: {model_results[best_model_name]['mean_score']:.4f}")

## 6. Hyperparameter Tuning

Now we'll perform hyperparameter tuning on the best model type using GridSearchCV.

In [ ]:
# Define hyperparameter grids for each model type
if TASK_TYPE == 'classification':
    param_grids = {
        'Decision Tree': {
            'model__max_depth': [3, 5, 7, 10, 15, None],
            'model__min_samples_split': [2, 5, 10, 20],
            'model__min_samples_leaf': [1, 2, 4, 8],
            'model__criterion': ['gini', 'entropy']
        },
        'Random Forest': {
            'model__n_estimators': [50, 100, 200],
            'model__max_depth': [5, 10, 15, None],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2', None]
        },
        'Gradient Boosting': {
            'model__n_estimators': [50, 100, 200],
            'model__learning_rate': [0.01, 0.1, 0.2],
            'model__max_depth': [3, 5, 7],
            'model__min_samples_split': [2, 5, 10],
            'model__subsample': [0.8, 0.9, 1.0]
        },
        'AdaBoost': {
            'model__n_estimators': [50, 100, 200, 300],
            'model__learning_rate': [0.01, 0.1, 0.5, 1.0]
        }
    }
else:
    param_grids = {
        'Decision Tree': {
            'model__max_depth': [3, 5, 7, 10, 15, None],
            'model__min_samples_split': [2, 5, 10, 20],
            'model__min_samples_leaf': [1, 2, 4, 8],
            'model__criterion': ['squared_error', 'friedman_mse', 'absolute_error']
        },
        'Random Forest': {
            'model__n_estimators': [50, 100, 200],
            'model__max_depth': [5, 10, 15, None],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2', None]
        },
        'Gradient Boosting': {
            'model__n_estimators': [50, 100, 200],
            'model__learning_rate': [0.01, 0.1, 0.2],
            'model__max_depth': [3, 5, 7],
            'model__min_samples_split': [2, 5, 10],
            'model__subsample': [0.8, 0.9, 1.0]
        },
        'AdaBoost': {
            'model__n_estimators': [50, 100, 200, 300],
            'model__learning_rate': [0.01, 0.1, 0.5, 1.0]
        }
    }

# Get the parameter grid for the best model
param_grid = param_grids[best_model_name]
print(f"Hyperparameter grid for {best_model_name}:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

In [ ]:
# Create pipeline with best model type
best_base_model = models[best_model_name]

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_base_model)
])

# Calculate total combinations
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)

print(f"Total hyperparameter combinations to search: {total_combinations}")

# Use RandomizedSearchCV if too many combinations, otherwise GridSearchCV
if total_combinations > 100:
    print("Using RandomizedSearchCV (sampling 50 combinations)...")
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        n_iter=50,
        cv=5,
        scoring=scoring,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
else:
    print("Using GridSearchCV (exhaustive search)...")
    search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
        verbose=1
    )

In [ ]:
# Perform hyperparameter search
print("Starting hyperparameter search...")
search.fit(X_train, y_train)
print("\nHyperparameter search complete!")

In [ ]:
# Display best parameters and score
print("Best Hyperparameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest Cross-Validation Score: {search.best_score_:.4f}")

if TASK_TYPE == 'regression':
    print(f"Best CV RMSE: {np.sqrt(-search.best_score_):.4f}")

In [ ]:
# View top 10 hyperparameter combinations
results_df = pd.DataFrame(search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

# Select relevant columns
display_cols = ['rank_test_score', 'mean_test_score', 'std_test_score'] + \
               [col for col in results_df.columns if col.startswith('param_model__')]

print("Top 10 Hyperparameter Combinations:")
results_df[display_cols].head(10)

## 7. Model Evaluation on Validation Set

In [ ]:
# Get the best model from the search
best_model = search.best_estimator_

# Make predictions on validation set
y_pred = best_model.predict(X_val)

In [ ]:
# Evaluate the model
if TASK_TYPE == 'classification':
    print("Classification Report:")
    print(classification_report(y_val, y_pred))
    
    print(f"\nValidation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
    print(f"Validation F1 Score: {f1_score(y_val, y_pred, average='weighted'):.4f}")
    
    # Confusion matrix
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()
    
else:
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    print(f"Validation Metrics:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R² Score: {r2:.4f}")
    
    # Actual vs Predicted plot
    plt.figure(figsize=(8, 6))
    plt.scatter(y_val, y_pred, alpha=0.5)
    plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title('Actual vs Predicted')
    plt.tight_layout()
    plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Extract feature importances from the trained model
model_step = best_model.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    # Get feature names after preprocessing
    preprocessor_fitted = best_model.named_steps['preprocessor']
    
    # Get feature names from the ColumnTransformer
    feature_names = []
    
    # Numerical features keep their names
    feature_names.extend(numerical_features)
    
    # Categorical features get one-hot encoded names
    if categorical_features:
        cat_encoder = preprocessor_fitted.named_transformers_['cat'].named_steps['onehot']
        cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
        feature_names.extend(cat_feature_names)
    
    # Create importance DataFrame
    importances = model_step.feature_importances_
    
    # Handle case where feature names might not match
    if len(feature_names) == len(importances):
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
    else:
        importance_df = pd.DataFrame({
            'Feature': [f'Feature_{i}' for i in range(len(importances))],
            'Importance': importances
        }).sort_values('Importance', ascending=False)
    
    # Plot top 20 features
    top_n = min(20, len(importance_df))
    plt.figure(figsize=(10, 8))
    plt.barh(importance_df['Feature'].head(top_n)[::-1], 
             importance_df['Importance'].head(top_n)[::-1],
             color='steelblue')
    plt.xlabel('Feature Importance')
    plt.ylabel('Feature')
    plt.title(f'Top {top_n} Feature Importances ({best_model_name})')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop {top_n} Most Important Features:")
    print(importance_df.head(top_n).to_string(index=False))
else:
    print("Feature importances not available for this model type.")

## 9. Visualize Decision Tree (if applicable)

In [ ]:
# Visualize the tree if it's a single decision tree
if best_model_name == 'Decision Tree':
    plt.figure(figsize=(20, 10))
    plot_tree(
        model_step,
        max_depth=3,  # Limit depth for readability
        filled=True,
        rounded=True,
        fontsize=10
    )
    plt.title('Decision Tree Visualization (max_depth=3)')
    plt.tight_layout()
    plt.show()
elif best_model_name == 'Random Forest':
    print("Visualizing the first tree from the Random Forest ensemble:")
    plt.figure(figsize=(20, 10))
    plot_tree(
        model_step.estimators_[0],
        max_depth=3,
        filled=True,
        rounded=True,
        fontsize=10
    )
    plt.title('First Tree from Random Forest (max_depth=3)')
    plt.tight_layout()
    plt.show()
else:
    print(f"Tree visualization not available for {best_model_name}.")

## 10. Generate Predictions on Test Set

In [ ]:
# Make predictions on the test set
X_test = test_df.copy()

# Remove target column from test set if it exists
if TARGET_COL in X_test.columns:
    X_test = X_test.drop(columns=[TARGET_COL])

# Generate predictions
test_predictions = best_model.predict(X_test)

print(f"Generated {len(test_predictions)} predictions on test set.")
print(f"\nPrediction distribution:")
print(pd.Series(test_predictions).describe())

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'id': range(len(test_predictions)),  # Adjust this based on your data
    TARGET_COL: test_predictions
})

# If classification with label encoder, convert back to original labels
if TASK_TYPE == 'classification' and 'label_encoder' in dir():
    submission[TARGET_COL] = label_encoder.inverse_transform(test_predictions)

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file saved to 'submission.csv'")
submission.head()

## 11. Summary and Conclusions

In [ ]:
print("="*60)
print("DECISION TREE PIPELINE - SUMMARY")
print("="*60)
print(f"\nTask Type: {TASK_TYPE.upper()}")
print(f"Best Model Type: {best_model_name}")
print(f"\nBest Hyperparameters:")
for param, value in search.best_params_.items():
    param_name = param.replace('model__', '')
    print(f"  - {param_name}: {value}")

print(f"\nCross-Validation Performance:")
if TASK_TYPE == 'classification':
    print(f"  - CV Accuracy: {search.best_score_:.4f}")
    print(f"  - Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
else:
    print(f"  - CV RMSE: {np.sqrt(-search.best_score_):.4f}")
    print(f"  - Validation RMSE: {rmse:.4f}")
    print(f"  - Validation R²: {r2:.4f}")

print(f"\nTest Predictions Generated: {len(test_predictions)}")
print("="*60)

## References

- [Scikit-learn Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [Scikit-learn Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [Scikit-learn Pipelines](https://scikit-learn.org/stable/modules/compose.html)
- [GridSearchCV Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)